import pandas as pd
import os
import csv

PASTA = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

# Schema canônico v08 — ordem e nomes corretos
COLUNAS_CANONICAS = [
    'fonte', 'data', 'item', 'pagina',
    'dimensao_ivs', 'codigo_variavel', 'tipo_relacao_variavel',
    'nivel_criticidade', 'resumo', 'origem', 'municipio_impactado',
    'tipo_abrangencia', 'tipo_evento', 'cod_loteamento',
    'nivel_confianca_loteamento', 'polaridade_evento', 'tipo_origem_evento',
    'observacao', 'nota_analista'
]

# Mapeamento de nomes antigos para nomes canônicos
RENOMEAR = {
    'data_publicacao'  : 'data',
    'titulo_evento'    : 'item',
    'resumo_evento'    : 'resumo',
    'localidade'       : 'origem',
    'gravidade'        : 'nivel_criticidade',
}

# ─────────────────────────────────────────────
# EXECUÇÃO
# ─────────────────────────────────────────────

relatorio = []

for arquivo in sorted(os.listdir(PASTA)):
    if not arquivo.endswith('.csv'):
        continue
    if arquivo == 'corpus_resumo_periodos_v07.csv':
        continue

    caminho = os.path.join(PASTA, arquivo)
    problemas = []

    # ── Detectar separador ──────────────────────────────────────────
    with open(caminho, encoding='utf-8') as f:
        primeira_linha = f.readline()

    if '\t' in primeira_linha:
        sep = '\t'
        # Remover aspas duplas extras coladas nos nomes das colunas
        df = pd.read_csv(caminho, sep='\t', dtype=str)
        df.columns = [c.strip().strip('"') for c in df.columns]
        problemas.append('separador_tab')
    else:
        df = pd.read_csv(caminho, dtype=str)

    # ── Renomear colunas antigas ────────────────────────────────────
    colunas_antes = list(df.columns)
    df = df.rename(columns=RENOMEAR)
    colunas_depois = list(df.columns)
    if colunas_antes != colunas_depois:
        problemas.append('colunas_renomeadas')

    # ── Garantir que todas as colunas canônicas existem ────────────
    for col in COLUNAS_CANONICAS:
        if col not in df.columns:
            df[col] = ''
            problemas.append(f'coluna_adicionada:{col}')

    # ── Remover colunas fora do schema canônico ────────────────────
    colunas_extras = [c for c in df.columns if c not in COLUNAS_CANONICAS]
    if colunas_extras:
        df = df.drop(columns=colunas_extras)
        problemas.append(f'colunas_removidas:{colunas_extras}')

    # ── Aplicar ordem canônica ──────────────────────────────────────
    df = df[COLUNAS_CANONICAS]

    # ── Remover linhas completamente vazias ────────────────────────
    linhas_antes = len(df)
    df = df.dropna(how='all')
    if len(df) < linhas_antes:
        problemas.append(f'linhas_vazias_removidas:{linhas_antes - len(df)}')

    # ── Gravar normalizado ─────────────────────────────────────────
    df.to_csv(caminho, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')

    status = '✅' if problemas else '  '
    relatorio.append((status, arquivo, problemas))

# ─────────────────────────────────────────────
# RELATÓRIO
# ─────────────────────────────────────────────

print(f"{'='*70}")
print(f"RELATÓRIO DE NORMALIZAÇÃO")
print(f"{'='*70}")
for status, arquivo, problemas in relatorio:
    if problemas:
        print(f"{status} {arquivo}")
        for p in problemas:
            print(f"     → {p}")
    else:
        print(f"   {arquivo} — sem alterações")

total_alterados = sum(1 for s, _, p in relatorio if p)
print(f"{'='*70}")
print(f"Total arquivos verificados : {len(relatorio)}")
print(f"Total arquivos alterados   : {total_alterados}")
print()

# ── Verificação final: concat limpo ───────────────────────────────
print("Verificando concat final...")
dfs = []
for arquivo in sorted(os.listdir(PASTA)):
    if not arquivo.endswith('.csv') or arquivo == 'corpus_resumo_periodos_v07.csv':
        continue
    caminho = os.path.join(PASTA, arquivo)
    df = pd.read_csv(caminho, dtype=str)
    dfs.append(df)

df_total = pd.concat(dfs, ignore_index=True)
suspeitas = df_total[df_total['fonte'].isna() | df_total['data'].isna()]

print(f"Total de registros : {len(df_total)}")
print(f"Linhas suspeitas   : {len(suspeitas)}")
if len(suspeitas) == 0:
    print()
    print('✅ Concat limpo — todos os arquivos normalizados com sucesso.')
else:
    print()
    print('⚠️  Ainda há linhas suspeitas — verificar manualmente.')
    print(suspeitas[['fonte', 'data', 'item', 'arquivo_origem']].head(10))


import pandas as pd
import os
import csv

PASTA = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

for arquivo in sorted(os.listdir(PASTA)):
    if not arquivo.endswith('.csv') or arquivo == 'corpus_resumo_periodos_v07.csv':
        continue
    caminho = os.path.join(PASTA, arquivo)
    df = pd.read_csv(caminho, dtype=str)
    suspeitas = df[df['fonte'].isna() | df['data'].isna()]
    if len(suspeitas) > 0:
        print(f"⚠️  {arquivo} — {len(suspeitas)} linha(s) suspeita(s) de {len(df)} total")
    else:
        print(f"   {arquivo} — ok")

import pandas as pd
import os
import csv

PASTA = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

dfs = []
for arquivo in sorted(os.listdir(PASTA)):
    if not arquivo.endswith('.csv') or arquivo == 'corpus_resumo_periodos_v07.csv':
        continue
    caminho = os.path.join(PASTA, arquivo)
    df = pd.read_csv(caminho, dtype=str)
    df["arquivo_origem"] = arquivo
    dfs.append(df)

df_total = pd.concat(dfs, ignore_index=True)

suspeitas = df_total[df_total['fonte'].isna() | df_total['data'].isna()]

print(f"Total de registros : {len(df_total)}")
print(f"Linhas suspeitas   : {len(suspeitas)}")

if len(suspeitas) == 0:
    print()
    print("✅ Corpus limpo — nenhuma linha suspeita.")
else:
    print()
    print("⚠️  Arquivos com problema:")
    print(suspeitas[['arquivo_origem', 'fonte', 'data', 'item']].to_string())

import pandas as pd
import os
import csv

PASTA = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"
caminho = os.path.join(PASTA, "2026_04_07_tribuna_liberal.csv")

df = pd.read_csv(caminho, dtype=str)
df["fonte"] = df["fonte"].fillna("Tribuna Liberal")
df.to_csv(caminho, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print("✅ fonte preenchida em 2026_04_07")
print(df[["fonte", "data", "item"]].to_string())